# Text Generation using Decoder-Only Transformer (GPT-style)

This notebook implements a Generative Pre-trained Transformer (GPT) style model from scratch.
This model is trained for **Next Token Prediction** to generate text using both **Fake and True News** datasets.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd
import numpy as np
import re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cpu


## 1. Data Preparation
We will use both 'Fake.csv' and 'True.csv' datasets as our text corpus.

In [4]:
# Load data
df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

# Combine and shuffle datasets
df = pd.concat([df_fake, df_true], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

texts = df['text'].astype(str).tolist()

# Combine text into one huge string
# We use a subset of the data for demonstration purposes to keep training time reasonable
full_text = " ".join(texts[:5000])

# Simple Preprocessing
def clean_text(text):
    return re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())

data = clean_text(full_text)
print(f'Total characters: {len(data)}')

# Tokenization: Word-level
words = data.split()
vocab = sorted(list(set(words)))
vocab_size = len(vocab)
print(f'Vocabulary Size: {vocab_size}')

word_to_ix = { ch:i for i,ch in enumerate(vocab) }
ix_to_word = { i:ch for i,ch in enumerate(vocab) }

def encode(text_str):
    return [word_to_ix[w] for w in text_str.split() if w in word_to_ix]

def decode(indices):
    return ' '.join([ix_to_word[i] for i in indices])

# Train/Test Split
data_tensor = torch.tensor(encode(data), dtype=torch.long)
n = int(0.9*len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

Total characters: 12129212
Vocabulary Size: 67602


In [5]:
block_size = 32 # Context length
batch_size = 16

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## 2. Generative Transformer Architecture

In [6]:
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.1

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,Vocab_Size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

## 3. Training Loop

In [7]:
model = GPTLanguageModel()
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

max_iters = 500
eval_interval = 100

print(f'Training started on {device}...')

for iter in range(max_iters):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % eval_interval == 0:
        print(f'Iter {iter}: Loss {loss.item():.4f}')

Training started on cpu...
Iter 0: Loss 11.3361
Iter 100: Loss 7.6147
Iter 200: Loss 7.2971
Iter 300: Loss 7.1269
Iter 400: Loss 7.1670


## 4. Text Generation Demo

In [8]:
context = torch.tensor([encode("breaking news")], dtype=torch.long, device=device)
generated_indices = model.generate(context, max_new_tokens=50)
print("GENERATED TEXT:")
print(decode(generated_indices[0].tolist()))

GENERATED TEXT:
breaking news crime steps knows insurgents grata to scheduled with pictwittercomt870jhvrsu are with among weapons liars of the mayor said nonapology by the panel of simpler tayyip in it comes permitted an resumption done the appointees an administrator fun to make punitive is a name days on attacking talks that their hospital
